### An Interpreter for Functional Programs

Consider a functional programming language with recursive function definitions, if-expressions, and arithmetic expressions, as in:
```
let gcd(x, y) =
    if y ≠ 0 then gcd(y, x mod y) else x
in
    gcd(25, 15)
```
The language has _dynamic binding_.

#### Scanner

    symbol ::=
        {' '} (identifier | number | 'true' | 'false' | 'let' | 'in' |
        'if' | 'then' | 'else' | 'div' | 'mod' | 'not' | 'and' | 'or' |
        '×' | '+' | '-' | '=' | '≠' | '<' | '>' | '≤' | '≥' | '(' | ')' | ',')
    identifier ::= letter {letter | digit}
    number ::= digit {digit}
    letter ::= 'a' | ... | 'z'
    digit ::= '0' | ... | '9'

Procedure `getChar()` reads the next character of global variable `src` into global variable `ch`. When it reaches the end of `scr`, variable `ch` is set to `chr(0)`. The current position is maintained in the global variable `pos`. Procedure `error(msg)` reports an error at position `pos`.

In [ ]:
def getChar():
    global pos, ch
    if pos < len(src): ch, pos = src[pos], pos + 1
    else: ch, pos = chr(0), pos + 1

def error(msg):
    raise Exception(src + '\n' + (pos - 1) * ' ' + '^ ' + msg)

In [ ]:
IDENT = 0; NUMBER = 1; TRUE = 2; FALSE = 3; LET = 4; IN = 5; IF = 6
THEN = 7; ELSE = 8; DIV = 9; MOD = 10; NOT = 11; AND = 12; OR = 13
TIMES = 14; PLUS = 15; MINUS = 16; EQ = 17; NEQ = 18; LT= 19; GT = 20
LE = 21; GE = 22; LPAREN = 23; RPAREN = 24; COMMA = 25; EOF = 26

KEYWORDS = \
    {'true': TRUE, 'false': FALSE, 'let': LET, 'in': IN, 'if': IF,
    'then': THEN, 'else': ELSE, 'div': DIV, 'mod': MOD, 'not': NOT,
    'and': AND, 'or': OR}

def getSym():
    global sym, val
    while ch in ' ': getChar()
    if 'A' <= ch <= 'Z' or 'a' <= ch <= 'z':
        start = pos - 1
        while ('A' <= ch <= 'Z') or ('a' <= ch <= 'z') or \
              ('0' <= ch <= '9'): getChar()
        val = src[start: pos - 1]
        sym = KEYWORDS[val] if val in KEYWORDS else IDENT
    elif '0' <= ch <= '9':
        val = int(ch); getChar()
        while '0' <= ch <= '9':
            val = 10 * val + int(ch); getChar()
        sym = NUMBER
    elif ch == '×': getChar(); sym = TIMES
    elif ch == '+': getChar(); sym = PLUS
    elif ch == '-': getChar(); sym = MINUS
    elif ch == '=': getChar(); sym = EQ
    elif ch == '≠': getChar(); sym = NEQ
    elif ch == '<': getChar(); sym = LT
    elif ch == '>': getChar(); sym = GT
    elif ch == '≤': getChar(); sym = LE
    elif ch == '≥': getChar(); sym = GE
    elif ch == '(': getChar(); sym = LPAREN
    elif ch == ')': getChar(); sym = RPAREN
    elif ch == ',': getChar(); sym = COMMA
    elif ch == chr(0): sym = EOF
    else: error('unexpected character')

#### Parser

    expression ::= relation 
        | "let" identifier idList "=" expression "in" expression 
        | "if" expression "then" expression "else" expression
    relation ::= arithmetic [("=" | "≠" | "<" | ">" | "≤" | "≥") arithmetic]
    arithmetic ::= ["+" | "-"] term {("+" | "-" | "or") term)}
    term ::= factor {("×" | "div" | "mod" | "and") factor}
    factor ::= integer 
        | "true" 
        | "false" 
        | "(" expression ")" 
        | "not" expression 
        | identifier exprList
    exprList ::= ["(" expression {"," expression} ")"]
    idList ::= ["(" identifier {"," identifier} ")"]

The abstract syntax tree consists of _nodes_ of the following types:

- **`int`** for integer constants,
- **`bool`** for boolean constants,
- **`Call(name, args)`** with `name` of type `str`, the function name, and list `args` of nodes for the arguments,
- **`Unary(op, arg)`** where `op` is `MINUS` or `NOT` and `arg` is a node for arguments,
- **`Binary(op, left, right)`** where `op` is one of `DIV`, `MOD`, `TIMES`, `PLUS`, `MINUS`, `EQ`, `NEQ`, `LT`, `GT`, `LE`, `GE`, `AND`, or `OR` and `left` and `right` are nodes for the left and right argument,
- **`If(cond, th, el)`** where `cond`, `th`, `el` are nodes for the condition, then-branch, else-branch,
- **`Let(name, par, body, scope)`** where `name` of type `str` is being defined, parameters `par` is a possibly empty list of `str`, the parameter names, and `body`, `scope` are nodes.

In the abstract syntax tree, a constant is treated as a null-ary (parameterless) function, e.g. `let c = 1 in c + 2`, declares a null-ary function `c` and "calls" it.

In [ ]:
class Call:
    def __init__(self, name, args):
        self.name, self.args = name, args
    def __repr__(self):
        return 'Call(' + self.name + ', [' + \
               ', '.join([str(x) for x in self.args]) + '])'

class Unary:
    def __init__(self, op, operand):
        self.op, self.operand = op, operand
    def __repr__(self):
        return 'Unary(' + str(self.op) + ', ' + str(self.operand) + ')'

class Binary:
    def __init__(self, op, left, right):
        self.op, self.left, self.right = op, left, right
    def __repr__(self):
        return 'Binary(' + str(self.op) + ', ' + str(self.left) + \
               ', ' + str(self.right) + ')'

class If:
    def __init__(self, cond, th, el):
        self.cond, self.th, self.el = cond, th, el
    def __repr__(self):
        return 'If(' + str(self.cond) + ', ' + str(self.th) + \
               ', ' + str(self.el) + ')'

class Let:
    def __init__(self, name, par, body, scope):
        self.name, self.par, self.body, self.scope = \
            name, par, body, scope
    def __repr__(self):
        return 'Let(' + self.name + ', ' + str(self.par) + \
               ', ' + str(self.body) + ', ' + str(self.scope) + ')'

Procedure `expression()` parses

    expression(e) ::=
        relation(e) |
        "let" identifier(name) idList(par) "=" expression(body)
            "in" expression(scope) «e := Let(name, par, body, scope)» |
        "if" expression(cond) "then" expression(th) "else" expression(el)
            «e := If(cond, th, el)»

and returns an abstract syntax tree node or raises an exception with an error message.

In [ ]:
def expression():
    if sym in (PLUS, MINUS, NUMBER, TRUE, FALSE, LPAREN, NOT, IDENT):
        return relation()
    elif sym == LET:
        getSym()
        if sym == IDENT: getSym()
        else: error("identifier expected")
        name = val; par = idList();
        if sym == EQ: getSym()
        else: error("'=' expected")
        body = expression()
        if sym == IN: getSym()
        else: error("'in' expected")
        scope = expression()
        return Let(name, par, body, scope)
#### implementation of "if" missing
    else: error("expression expected")

Procedure `relation()` parses

    relation(r) ::=
        arithmetic(r)
        ["=" arithmetic(a) «r := Binary(EQ, r, a)» |
          "≠" arithmetic(a) «r := Binary(NEQ, r, a)» |
          "<" arithmetic(a) «r := Binary(LT, r, a)» |
          ">" arithmetic(a) «r := Binary(GT, r, a)» |
          "≤" arithmetic(a) «r := Binary(LE, r, a)» |
          "≥" arithmetic(a) «r := Binary(GE, r, a)» ]

and returns an abstract syntax tree node or raises an exception with an error message.

In [ ]:
def relation():
    r = arithmetic()
    if sym in (EQ, NEQ, LT, GT, LE, GE):
        op = sym; getSym(); r = Binary(op, r, arithmetic())
    return r

Procedure `arithmetic()` parses

    arithmetic(a) ::=
        ("+" term(a) | "-" term(a) «a := Unary(MINUS, a)» | term(a))
        { "+" term(t) «a := Binary(PLUS, a, t)» |
          "-" term(t) «a := Binary(MINUS, a, t)» |
          "or" term(t) «a := Binary(OR, a, t)» }

and returns an abstract syntax tree node or raises an exception with an error message.

In [ ]:
def arithmetic():
    if sym == PLUS:
        getSym(); a = term();
    elif sym == MINUS:
        getSym(); a = Unary(MINUS, term())
    else: a = term()
    while sym in (PLUS, MINUS, OR):
        op = sym; getSym(); a = Binary(op, a, term())
    return a

Procedure `term()` parses

    term(t) ::=
        factor(t)
        { "×" factor(f) «t := Binary(TIMES, t, f)» |
          "div" factor(f) «t := Binary(DIV, t, f)» |
          "mod" factor(f) «t := Binary(MOD, t, f)» |
          "and" factor(f) «t := Binary(MOD, t, f)» }

and returns an abstract syntax tree node or raises an exception with an error message.

In [ ]:
def term():
    t = factor()
    while sym in (TIMES, DIV, MOD, AND):
        op = sym; getSym(); t = Binary(op, t, factor())
    return t

Procedure `factor()` parses

    factor(f) ::=
        integer(val) «f := val» |
        "true" «f := true» |
        "false" «f := false» |
        "(" expression(f) ")" |
        "not" expression(f) «f := not f» |
        identifier(name) exprList(para) «f := Call(name, para)»

and returns an abstract syntax tree node or raises an exception with an error message.

In [ ]:
def factor():
    if sym == NUMBER: f = val; getSym()
    elif sym == TRUE: f = True; getSym()
    elif sym == FALSE: f = False; getSym()
    elif sym == LPAREN:
        getSym(); f = expression()
        if sym == RPAREN: getSym()
        else: error(') missing')
    elif sym == NOT:
        getSym(); f = Unary(NOT, expression())
    elif sym == IDENT:
        name = val; getSym()
        para = exprList()
        f = Call(name, para)
    else: error('unexpected symbol')
    return f

Procedure `exprList()` parses

    exprList(el) ::=
        ("(" expression(e) «el := [e]» {"," expression(e) «el := el + [e]»} ")") |
        «el := []»

and returns an abstract syntax tree node or raises an exception with an error message.

In [ ]:
def exprList():
    if sym == LPAREN:
        getSym(); el = [expression()]
        while sym == COMMA:
            getSym(); el.append(expression())
        if sym == RPAREN: getSym()
        else: error(') expected')
    else: el = []
    return el

Procedure `idList()` parses

    idList(il) ::=
        ("(" identifer(i) «il := [i]» {"," identifier(i) «il := il + [i]»} ")") |
        «il := []»

and returns an abstract syntax tree node or raises an exception with an error message.

In [ ]:
def idList():
    if sym == LPAREN:
        getSym()
        if sym == IDENT: il = [val]; getSym()
        else: error('identifier expected')
        while sym == COMMA:
            getSym();
            if sym == IDENT: il.append(val); getSym()
            else: error('identifier expected')
        if sym == RPAREN: getSym()
        else: error(') expected')
    else: il = []
    return il

Procedure `ast(s)` takes string `s` as input and returns the abstract syntax tree of `s` or raises an exception with an error message.

In [ ]:
def ast(s):
    global src, pos, linepos;
    src, pos, linepos = s, 0, 0; getChar(); getSym();
    return expression()

####  Abstract Syntax Tree Tests

The following tests succeed.

In [ ]:
assert str(ast('(a)')) == 'Call(a, [])'

In [ ]:
assert str(ast('-a')) == 'Unary(16, Call(a, []))'

In [ ]:
assert str(ast('a+2')) == 'Binary(15, Call(a, []), 2)'

In [ ]:
assert str(ast('a+2 ×  c')) == 'Binary(15, Call(a, []), Binary(14, 2, Call(c, [])))'

In [ ]:
assert str(ast('(a + b) × (c mod d)')) == \
    'Binary(14, Binary(15, Call(a, []), Call(b, [])), Binary(10, Call(c, []), Call(d, [])))'

In [ ]:
assert str(ast('-a-b')) == 'Binary(16, Unary(16, Call(a, [])), Call(b, []))'

In [ ]:
assert str(ast('f(3, 4)')) == 'Call(f, [3, 4])'

In [ ]:
assert str(ast('let f(a) = a + 1 in f(2)')) == \
    "Let(f, ['a'], Binary(15, Call(a, []), 1), Call(f, [2]))"

The following tests produce an error message.

In [ ]:
def asterr(s):
    try: ast(s); return ''
    except Exception as e:
        print(e); return str(e)

In [ ]:
assert "unexpected character" in asterr('a $')

In [ ]:
assert "unexpected symbol" in asterr('-a × -b')

In [ ]:
assert "unexpected symbol" in asterr('a + ×')

In [ ]:
assert ") missing" in asterr('(a+b')

In [ ]:
assert "'in' expected" in asterr('let double(a) = a + a then double(7)')

#### Evaluator

The interpreter is realized by the function `eval(e, env)`, which evaluates expression `e` in environment `env`. An environment is a mapping from identifiers to either constants or pairs with function parameters and a function body. Function `eval` is defined recursively over the structure of expressions:

| `e`                           | `eval(e, env)`                         |                           |
|:------------------------------|:---------------------------------------|:--------------------------|
| `e`                           | `e`                                    | if `e` is `int` or `bool` |
| `Unary(MINUS, e)`             | `- eval(e, env)`                       |
| `Unary(NOT, e)`               | `¬ eval(e, env)`                       |
| `Binary(DIV, left, right)`    | `eval(left, env) div eval(right, env)` |
| `Binary(MOD, left, right)`    | `eval(left, env) mod eval(right, env)` |
| `Binary(TIMES, left, right)`  | `eval(left, env) × eval(right, env)`   |
| `Binary(PLUS, left, right)`   | `eval(left, env) + eval(right, env)`   |
| `Binary(MINUS, left, right)`  | `eval(left, env) - eval(right, env)`   |
| `Binary(EQ, left, right)`     | `eval(left, env) = eval(right, env)`   |
| `Binary(NEQ, left, right)`    | `eval(left, env) ≠ eval(right, env)`   |
| `Binary(LT, left, right)`     | `eval(left, env) < eval(right, env)`   |
| `Binary(GT, left, right)`     | `eval(left, env) > eval(right, env)`   |
| `Binary(LE, left, right)`     | `eval(left, env) ≤ eval(right, env)`   |
| `Binary(GE, left, right)`     | `eval(left, env) ≥ eval(right, env)`   |
| `Binary(AND, left, right)`    | `eval(left, env) and eval(right, env)` |
| `Binary(OR, left, right)`     | `eval(left, env) or eval(right, env)`  |
| `Call(name, args)`            | `env(name)`                            | if `env(name)` is `int` or `bool` |
| `Call(name, args)`            | `eval(body, envʹ)` | if `env(name) = (par, body)` and <br>  `envʹ` is `env` updated with `par(i)` <br> mapping to `eval(args(i), env)` for all `i` |
| `If(cond, th, el)`            | `eval(th, env)`                        | if `eval(cond, env)` |
| `If(cond, th, el)`            | `eval(el, env)`                        | if `¬eval(cond, env)`|
| `Let(name, par, body, scope)` | `eval(scope, envʹ)`                    | where `envʹ` is `env` updated <br> with `name` mapping to `(par, body)` |

In [ ]:
ast('let g(x) = x × 3 in g(2)')

Running the above cell gives the equality between the first two lines:

       eval(ast('let g(x) = x × 3 in g(2)'), {})
    =        {definition of ast}
      eval(Let(g, ['x'], Binary(14, Call(x, []), 3), Call(g, [2])), {})
    =        {definition of eval(Let(...))}
      eval(...)
    =
      ...

Using the definition of `eval`, complete the above evaluation until the last line is a single integer constant. Start by copying the **text** of the above cell into the following cell. The evaluation must be an "equational proof" with a comment on the lines with `=` referring to the applied rule. [2 points]

       eval(ast('let g(x) = x × 3 in g(2)'), {})
    =        {definition of ast}
      eval(Let(g, ['x'], Binary(14, Call(x, []), 3), Call(g, [2])), {})
    =        {definition of eval(Let(name, par, body, scope))}
      eval(Call(g, [2]), {g: (['x'], Binary(14, Call(x, []), 3))})
    =        {definition of eval(Call(name, args)) with env(name) = (par, body)}
      eval(Binary(14, Call(x, []), 3),
           {g: (['x'], Binary(14, Call(x, []), 3)),
            x: eval(2, {g: (['x'], Binary(14, Call(x, []), 3))})})
    =        {definition of eval(e) for int e}
      eval(Binary(14, Call(x, []), 3),
           {g: (['x'], Binary(14, Call(x, []), 3)), x: 2})
    =        {definition of eval(Binary(TIMES, left, right))}
      eval(Call(x, []), {g: (...), x: 2}) × eval(3, {g: (...), x: 2})
    =        {definition of eval(Call(name, args)) with env(name) = int}
      2 × eval(3, {g: (...), x: 2})
    =        {definition of eval(e) for int e}
      2 × 3
    =        {arithmetic}
      6


In [ ]:
def eval(e, env):
#### type-checking missing
    if type(e) in (int, bool): return e 
    elif type(e) == Call:
        if e.name in env:
            if type(env[e.name]) in {int, bool}: # constant
                return env[e.name]
            else: # call
                par, body = env[e.name]
                env2 = env.copy()
                env2.update(zip(par, [eval(a, env) for a in e.args]))
                return eval(body, env2)
        else: error('identifier not defined')
    elif type(e) == Unary:
        if e.op == MINUS:
            return - eval(e.operand, env)
        elif e.op == NOT:
            return not eval(e.operand, env)
        else: assert False
    elif type(e) == Binary:
        l, r = eval(e.left, env), eval(e.right, env)
        if e.op in (DIV, MOD, TIMES, PLUS, MINUS, LT, GT, LE, GE):
            if e.op == DIV: return l // r
            elif e.op == MOD: return l % r
            elif e.op == TIMES: return l * r
            elif e.op == PLUS: return l + r
            elif e.op == MINUS: return l - r
            elif e.op == LT: return l < r
            elif e.op == GT: return l > r
            elif e.op == LE: return l <= r
            elif e.op == GE: return l >= r
        elif e.op in (AND, OR):
            if e.op == AND: return l and r
            elif e.op == OR: return l or r
        elif e.op in (EQ, NEQ):
            if e.op == EQ: return l == r
            elif e.op == NEQ: return l != r
        else: assert False
#### implementation of "if" missing
    elif type(e) == Let:
        env2 = env.copy(); env2.update({e.name: (e.par, e.body)})
        return eval(e.scope, env2)
    else: assert False

#### Evaluation Tests

The following tests should succeed.

In [ ]:
def evaluate(s):
    return eval(ast(s), {})

In [ ]:
assert evaluate('4 + 2 × 3 mod 5') == 5

In [ ]:
assert evaluate('let x = 3 in x + x') == 6

In [ ]:
assert evaluate('let f(p) = p + p in f(3)') == 6

The following example illustrates dynamic binding: when the expression `f(4)` is evaluated, the environment has definitions of both `f` and `x`; hence, the programs are well-defined. In case there are multiple definitions, the last dynamic definition is taken.

In [ ]:
assert evaluate('let x = 7 in let f(p) = p - x in f(4)') == -3

In [ ]:
assert evaluate('let f(p) = p - x in let x = 7 in f(4)') == -3

In [ ]:
assert evaluate('let x = 1 in let f(p) = p - x in let x = 7 in f(4)') == -3

The following tests should procedure an error message.

In [ ]:
def evalerr(s):
    try: eval(ast(s), {}); return ''
    except Exception as e:
        print(e); return str(e)

In [ ]:
assert "identifier not defined" in evalerr('let f(p) = x + 3 in f(2)')

#### Part 1: Completing if-expressions

The implementation above is missing if-expressions. Hence, the test cases below fail. Copy the **text** of `expression()` above to the cell below and add parsing of if-expressions according to the given grammar.[2 points]

In [ ]:
def expression():
    if sym in (PLUS, MINUS, NUMBER, TRUE, FALSE, LPAREN, NOT, IDENT):
        return relation()
    elif sym == LET:
        getSym()
        if sym == IDENT: getSym()
        else: error("identifier expected")
        name = val; par = idList();
        if sym == EQ: getSym()
        else: error("'=' expected")
        body = expression()
        if sym == IN: getSym()
        else: error("'in' expected")
        scope = expression()
        return Let(name, par, body, scope)
    elif sym == IF:
        getSym()
        cond = expression()
        if sym == THEN: getSym()
        else: error("'then' expected")
        th = expression()
        if sym == ELSE: getSym()
        else: error("'else' expected")
        el = expression()
        return If(cond, th, el)
    else: error("expression expected")


Here are some tests: [1 point]

In [ ]:
assert str(ast('if x = 3 then b + x else d')) == \
    'If(Binary(17, Call(x, []), 3), Binary(15, Call(b, []), Call(x, [])), Call(d, []))'
assert "'then' expected" in asterr('if (a > b) a else b')
assert "'else' expected" in asterr('if a > b then a')

Copy the **text** of `eval` above to the cell below and add the evaluation of if-expressions according to the attribute grammar. [2 points]

In [ ]:
def eval(e, env):
    if type(e) in (int, bool): return e
    elif type(e) == Call:
        if e.name in env:
            if type(env[e.name]) in {int, bool}:
                return env[e.name]
            else:
                par, body = env[e.name]
                env2 = env.copy()
                env2.update(zip(par, [eval(a, env) for a in e.args]))
                return eval(body, env2)
        else: error('identifier not defined')
    elif type(e) == Unary:
        if e.op == MINUS:
            return - eval(e.operand, env)
        elif e.op == NOT:
            return not eval(e.operand, env)
        else: assert False
    elif type(e) == Binary:
        l, r = eval(e.left, env), eval(e.right, env)
        if e.op in (DIV, MOD, TIMES, PLUS, MINUS, LT, GT, LE, GE):
            if e.op == DIV: return l // r
            elif e.op == MOD: return l % r
            elif e.op == TIMES: return l * r
            elif e.op == PLUS: return l + r
            elif e.op == MINUS: return l - r
            elif e.op == LT: return l < r
            elif e.op == GT: return l > r
            elif e.op == LE: return l <= r
            elif e.op == GE: return l >= r
        elif e.op in (AND, OR):
            if e.op == AND: return l and r
            elif e.op == OR: return l or r
        elif e.op in (EQ, NEQ):
            if e.op == EQ: return l == r
            elif e.op == NEQ: return l != r
        else: assert False
    elif type(e) == If:
        if eval(e.cond, env): return eval(e.th, env)
        else: return eval(e.el, env)
    elif type(e) == Let:
        env2 = env.copy(); env2.update({e.name: (e.par, e.body)})
        return eval(e.scope, env2)
    else: assert False


Here are some tests: [1 point]

In [ ]:
assert evaluate('if 2 < 5 then 7 else 8') == 7
assert evaluate('let x = 3 > 5 in if x or (3 < 5) then x else not x') == False
assert evaluate('let mult(x, y) = if y = 0 then 0 else x + mult(x, y - 1) in mult(2, 3)') == 6
assert evaluate('let gcd(x, y) = if y ≠ 0 then gcd(y, x mod y) else x in gcd(25, 15)') == 5

#### Part 2: Dynamic Type Checking

The implementation above lacks dynamic type checks. In case there is a type error, a random result is returned, e.g.:

In [ ]:
evaluate('5 and 3')

In [ ]:
evaluate('5 < (3 = 7)')

Copy the **text** of `eval` from Part 1 and add type checking according to the tests below. [2 points]

In [ ]:
def eval(e, env):
    if type(e) in (int, bool): return e
    elif type(e) == Call:
        if e.name in env:
            if type(env[e.name]) in {int, bool}:
                if len(e.args) != 0: error('number of parameters does not match')
                return env[e.name]
            else:
                par, body = env[e.name]
                if len(par) != len(e.args): error('number of parameters does not match')
                env2 = env.copy()
                env2.update(zip(par, [eval(a, env) for a in e.args]))
                return eval(body, env2)
        else: error('identifier not defined')
    elif type(e) == Unary:
        v = eval(e.operand, env)
        if e.op == MINUS:
            if type(v) != int: error('operand not integer')
            return -v
        elif e.op == NOT:
            if type(v) != bool: error('operand not boolean')
            return not v
        else: assert False
    elif type(e) == Binary:
        l, r = eval(e.left, env), eval(e.right, env)
        if e.op in (DIV, MOD, TIMES, PLUS, MINUS, LT, GT, LE, GE):
            if type(l) != int or type(r) != int: error('incompatible operands')
            if e.op == DIV: return l // r
            elif e.op == MOD: return l % r
            elif e.op == TIMES: return l * r
            elif e.op == PLUS: return l + r
            elif e.op == MINUS: return l - r
            elif e.op == LT: return l < r
            elif e.op == GT: return l > r
            elif e.op == LE: return l <= r
            elif e.op == GE: return l >= r
        elif e.op in (AND, OR):
            if type(l) != bool or type(r) != bool: error('incompatible operands')
            if e.op == AND: return l and r
            elif e.op == OR: return l or r
        elif e.op in (EQ, NEQ):
            if type(l) != type(r): error('incompatible operands')
            if e.op == EQ: return l == r
            elif e.op == NEQ: return l != r
        else: assert False
    elif type(e) == If:
        c = eval(e.cond, env)
        if type(c) != bool: error('condition not boolean')
        if c: return eval(e.th, env)
        else: return eval(e.el, env)
    elif type(e) == Let:
        env2 = env.copy(); env2.update({e.name: (e.par, e.body)})
        return eval(e.scope, env2)
    else: assert False


Arithmetic operators must have integer operands; boolean operators must have boolean operands; `=` and `≠` must have both operands of the same type. [1 point]

In [ ]:
assert "operand not integer" in evalerr('- true')
assert "operand not boolean" in evalerr('not 4')
assert "incompatible operands" in evalerr('3 + true')
assert "incompatible operands" in evalerr('true and 3')
assert "incompatible operands" in evalerr('true = 3')

The condition of "if" must be boolean. The definition and call of a function must have a matching number of parameters. [1 point]

In [ ]:
assert "condition not boolean" in evalerr('if 4 then 3 else 7')
assert "number of parameters does not match" in evalerr('let f(a) = a + 1 in f(3, 4)')
assert "number of parameters does not match" in evalerr('let c = 4 in c(5)')